In [2]:
import pandas as pd

print("🏀 INITIALIZING TOURNAMENT ORACLE... 🏀")

# Load your final combined submission file
try:
    submission = pd.read_csv('submission.csv')
    m_teams = pd.read_csv('MTeams.csv')
    w_teams = pd.read_csv('WTeams.csv')
except FileNotFoundError:
    print("[!] Error: Make sure 'submission.csv', 'MTeams.csv', and 'WTeams.csv' are in the directory.")

# Dictionaries for looking up IDs by spelling
m_name_to_id = dict(zip(m_teams['TeamName'].str.lower().str.strip(), m_teams['TeamID']))
w_name_to_id = dict(zip(w_teams['TeamName'].str.lower().str.strip(), w_teams['TeamID']))

# Dictionary for mapping IDs back to beautiful names
id_to_name = {**dict(zip(m_teams['TeamID'], m_teams['TeamName'])),
              **dict(zip(w_teams['TeamID'], w_teams['TeamName']))}

def get_matchup_prediction(team_a_name, team_b_name, tournament='M', season=2026):
    """
    Looks up the exact probability of Team A beating Team B.
    Set tournament to 'M' for Men's bracket, or 'W' for Women's bracket.
    """
    team_a_lower = team_a_name.lower().strip()
    team_b_lower = team_b_name.lower().strip()

    lookup_dict = m_name_to_id if tournament.upper() == 'M' else w_name_to_id

    if team_a_lower not in lookup_dict:
        return f"[!] Error: Could not find '{team_a_name}'. Check spelling!"
    if team_b_lower not in lookup_dict:
        return f"[!] Error: Could not find '{team_b_name}'. Check spelling!"

    team_a_id = lookup_dict[team_a_lower]
    team_b_id = lookup_dict[team_b_lower]

    # Kaggle IDs must be ordered with the smaller ID first
    if team_a_id < team_b_id:
        kaggle_id = f"{season}_{team_a_id}_{team_b_id}"
        is_team_a_first = True
    else:
        kaggle_id = f"{season}_{team_b_id}_{team_a_id}"
        is_team_a_first = False

    match = submission[submission['ID'] == kaggle_id]

    if match.empty:
        return f"[!] Error: Matchup {kaggle_id} not found in the submission file."

    raw_prob = match['Pred'].iloc[0]
    team_a_win_prob = raw_prob if is_team_a_first else (1.0 - raw_prob)
    team_b_win_prob = 1.0 - team_a_win_prob

    if team_a_win_prob > team_b_win_prob:
        favorite, fav_prob = id_to_name[team_a_id], team_a_win_prob
        underdog, und_prob = id_to_name[team_b_id], team_b_win_prob
    else:
        favorite, fav_prob = id_to_name[team_b_id], team_b_win_prob
        underdog, und_prob = id_to_name[team_a_id], team_a_win_prob

    bracket_label = "Men's" if tournament.upper() == 'M' else "Women's"

    print(f"\n==============================================")
    print(f" 🔮 {bracket_label} PREDICTION: {id_to_name[team_a_id]} vs. {id_to_name[team_b_id]}")
    print(f"==============================================")
    print(f" ⭐ Favorite:  {favorite} ({fav_prob * 100:.1f}%)")
    print(f" 🐕 Underdog:  {underdog} ({und_prob * 100:.1f}%)")
    print(f"==============================================\n")

print("✅ Oracle Ready! Use the cell below to check games.")

🏀 INITIALIZING TOURNAMENT ORACLE... 🏀
✅ Oracle Ready! Use the cell below to check games.


In [3]:
# Look up a Men's game
get_matchup_prediction("Duke", "North Carolina", tournament='M', season=2026)

# Look up a Women's game
get_matchup_prediction("South Carolina", "Iowa", tournament='W', season=2026)


 🔮 Men's PREDICTION: Duke vs. North Carolina
 ⭐ Favorite:  Duke (83.4%)
 🐕 Underdog:  North Carolina (16.6%)


 🔮 Women's PREDICTION: South Carolina vs. Iowa
 ⭐ Favorite:  Iowa (52.1%)
 🐕 Underdog:  South Carolina (47.9%)

